# Debug undefined initial likelihood

Build a country/analysis the same way as `run_single_country`, then inspect NUTS initialisation **without sampling**.

If the eight progress bars in `02-run.ipynb` sit at 0% and finish immediately, the initial potential energy is NaN or -inf.
Typical causes here: `log` of modelled counts that are 0, or `prop_* = strain_inc / total_inc` when total incidence is 0.

`beta` is Uniform[`BETA_MIN`, `BETA_MAX`]. Values outside that interval make the prior density -inf.

In [1]:
from datetime import timedelta

import pandas as pd
from jax import numpy as jnp, random
from numpyro.distributions import HalfNormal
from numpyro.infer.util import initialize_model, log_density

from emu_renewal.calibration import StandardCalib
from emu_renewal.constants import (
    BETA_INIT_OC,
    BETA_MAX,
    BETA_MIN,
    DATA_PATH,
    INDEP_EFFECT_INIT,
    RUN_DATA_DELAY,
)
from emu_renewal.indicators import (
    get_alpha_info,
    get_ba2_info,
    get_ba5_info,
    get_cases_target,
    get_country_vars,
    get_deaths_target,
    get_delta_info,
    get_hosp_target,
    get_seroprev_target,
)
from emu_renewal.inputs import get_country_pop, store_oxcgrt_data
from emu_renewal.priors import get_standard_priors
from emu_renewal.renew import MultiStrainModel
from emu_renewal.run import (
    find_run_end_time,
    find_run_start_time,
    get_scaler_provider,
    jax_config_cpu_only,
)
from emu_renewal.utils import get_cont_of_country, to_iso3

jax_config_cpu_only()

from emu_renewal.calibration import StandardCalib
from emu_renewal.constants import (
    BETA_INIT_OC,
    BETA_MAX,
    BETA_MIN,
    DATA_PATH,
    INDEP_EFFECT_INIT,
    RUN_DATA_DELAY,
)
from emu_renewal.indicators import (
    get_alpha_info,
    get_ba2_info,
    get_ba5_info,
    get_cases_target,
    get_country_vars,
    get_deaths_target,
    get_delta_info,
    get_hosp_target,
    get_seroprev_target,
)
from emu_renewal.inputs import get_country_pop, store_oxcgrt_data
from emu_renewal.priors import get_standard_priors
from emu_renewal.renew import MultiStrainModel
from emu_renewal.run import (
    find_run_end_time,
    find_run_start_time,
    get_scaler_provider,
    jax_config_cpu_only,
)
from emu_renewal.utils import get_cont_of_country, to_iso3

jax_config_cpu_only()

In [2]:
iso3 = "SGP"
analyses = ["no_scaling", "oxcgrt_floored", "oxcgrt_independent"]
beta_init = BETA_INIT_OC
iso3, analyses, beta_init, (BETA_MIN, BETA_MAX)

('SGP',
 ['no_scaling', 'oxcgrt_floored', 'oxcgrt_independent'],
 1.0,
 (0.5, 3.5))

In [3]:
def build(iso3, analysis_type):
    iso3 = to_iso3(iso3)
    continent = get_cont_of_country(iso3)
    restriction_path = DATA_PATH / "restrictions"
    restriction_path.mkdir(exist_ok=True)
    if not (restriction_path / "oxcgrt.csv").exists():
        store_oxcgrt_data()

    pop = get_country_pop(iso3)
    data_start = find_run_start_time(pop, iso3)
    end_time = find_run_end_time(iso3)
    run_start = data_start - timedelta(RUN_DATA_DELAY)

    n_deaths, deaths_targ = get_deaths_target(iso3, data_start, end_time)
    cases_targ = get_cases_target(iso3, data_start, end_time, n_deaths)
    hosp_targ = get_hosp_target(iso3, data_start, end_time, n_deaths)
    seroprev_targ = get_seroprev_target(iso3, continent, data_start, end_time)

    var_data = get_country_vars(iso3)
    delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
    alpha_var, alpha_targ, alpha_seed = get_alpha_info(
        iso3, var_data, continent, end_time, delta_targ
    )
    ba2_var, ba2_targ, ba2_seed = get_ba2_info(var_data, continent)
    ba5_var, ba5_targ, ba5_seed = get_ba5_info(var_data, continent)
    start_var = "ba1" if continent == "OC" else "eu"
    var_names = [start_var] + alpha_var + delta_var + ba2_var + ba5_var
    seed_times = alpha_seed + delta_seed + ba2_seed + ba5_seed
    var_targs = alpha_targ | delta_targ | ba2_targ | ba5_targ

    scaler_provider = get_scaler_provider(iso3, analysis_type)
    if scaler_provider.ts_end:
        end_time = min([end_time, scaler_provider.ts_end])

    model = MultiStrainModel(
        pop, run_start, end_time, var_names, seed_times, scaler_provider, continent == "OC"
    )
    hosp_key = list(hosp_targ.keys())[0] if hosp_targ else ""
    priors = get_standard_priors(len(var_names), hosp_key, iso3, continent) | scaler_provider.get_priors()
    targets = deaths_targ | cases_targ | hosp_targ | seroprev_targ | var_targs
    calib = StandardCalib(model, priors, targets)
    return {
        "iso3": iso3,
        "analysis": analysis_type,
        "pop": pop,
        "run_start": run_start,
        "data_start": data_start,
        "end_time": end_time,
        "var_names": var_names,
        "active_targets": calib.active_targets,
        "model": model,
        "calib": calib,
    }


def init_params(calib, beta=None):
    if beta is None:
        beta = beta_init
    params = dict(calib.fixed_params)
    for name, prior in calib.sampled_params.items():
        if name == "beta":
            params[name] = jnp.array(beta)
        elif name == "ts_weights" and isinstance(prior, HalfNormal):
            params[name] = jnp.full(prior.batch_shape, INDEP_EFFECT_INIT)
        else:
            params[name] = jnp.asarray(prior.mean)
    params["dispersion_proc"] = jnp.asarray(calib.proc_dispersion.mean)
    params["proc"] = jnp.zeros(calib.n_proc_periods)
    return params


def latent_params(calib, params):
    names = list(calib.sampled_params) + ["dispersion_proc", "proc"]
    return {k: params[k] for k in names}


def array_report(x):
    x = jnp.asarray(x)
    finite = x[jnp.isfinite(x)]
    return {
        "min": float(finite.min()) if finite.size else None,
        "max": float(finite.max()) if finite.size else None,
        "n_nan": int(jnp.isnan(x).sum()),
        "n_inf": int(jnp.isinf(x).sum()),
        "n_le0": int((x <= 0).sum()),
        "n": int(x.size),
    }


def inspect_result(model, params, calib):
    result = model.renewal_func(**params)
    scale = model.scaler.get_parameterised_scaler(**params)
    keys = ["weekly_cases", "weekly_deaths", "prop_ba2", "prop_ba5", "prop_delta", "prop_alpha"]
    series = {k: array_report(result[k]) for k in keys if k in result}
    series["scale"] = array_report(scale)
    series["scale_t0"] = float(scale[0])
    target_ll = {}
    for ind in calib.active_targets:
        modelled = result[ind][calib.common_idx[ind]]
        target_ll[ind] = {
            **array_report(modelled),
            "ll": float(calib.targets[ind].loglikelihood(modelled, params)),
        }
    return result, scale, series, target_ll

In [4]:
if not (BETA_MIN <= beta_init <= BETA_MAX):
    print(f"beta_init={beta_init} is outside Uniform({BETA_MIN}, {BETA_MAX})")

rows = []
for analysis in analyses:
    built = build(iso3, analysis)
    model, calib = built["model"], built["calib"]
    params = init_params(calib)
    result, scale, series, target_ll = inspect_result(model, params, calib)
    joint, _ = log_density(calib.calibration, (), {}, latent_params(calib, params))
    try:
        nuts_init = initialize_model(
            random.PRNGKey(3),
            calib.calibration,
            init_strategy=calib.init_strategy,
        )
        nuts_pe = float(nuts_init.param_info.potential_energy)
    except Exception as e:
        nuts_pe = repr(e)
    print(f"\n=== {iso3} {analysis} ===")
    print(
        f"window {built['run_start'].date()} -> {built['end_time'].date()} "
        f"(data {built['data_start'].date()}), strains {built['var_names']}"
    )
    print(f"log_density at constructed init: {float(joint)}")
    print(f"NUTS initial potential energy:   {nuts_pe}")
    print("outputs:", pd.DataFrame(series).T)
    print("target modelled / ll:", pd.DataFrame(target_ll).T)
    rows.append(
        {
            "analysis": analysis,
            "log_density": float(joint),
            "nuts_pe": nuts_pe,
            "scale_t0": series["scale_t0"],
            "cases_nan": series["weekly_cases"]["n_nan"],
            "deaths_nan": series["weekly_deaths"]["n_nan"],
            "cases_le0": series["weekly_cases"]["n_le0"],
            "deaths_le0": series["weekly_deaths"]["n_le0"],
        }
    )

pd.DataFrame(rows).set_index("analysis")


=== SGP no_scaling ===
window 2021-12-06 -> 2022-12-31 (data 2022-01-25), strains ['ba1', 'ba2', 'ba5']
log_density at constructed init: -5267.433289592132
NUTS initial potential energy:   6504.400678101884
outputs:                         min            max  n_nan  n_inf  n_le0      n
weekly_cases   7.355796e-10  398108.262482    0.0    0.0    0.0  391.0
weekly_deaths  1.869363e-55    2625.197874    0.0    0.0    0.0  391.0
prop_ba2       0.000000e+00       0.000000    0.0    0.0  391.0  391.0
prop_ba5       0.000000e+00       1.000000    0.0    0.0   80.0  391.0
scale          1.000000e+00       1.000000    0.0    0.0    0.0  391.0
scale_t0       1.000000e+00       1.000000    1.0    1.0    1.0    1.0
target modelled / ll:                         min            max  n_nan  n_inf  n_le0     n  \
weekly_deaths  1.036944e-10    2569.245410    0.0    0.0    0.0  48.0   
weekly_cases   1.682509e-09  388948.559807    0.0    0.0    0.0  48.0   
prop_ba2       0.000000e+00       0.000000   

,log_density,nuts_pe,scale_t0,cases_nan,deaths_nan,cases_le0,deaths_le0
analysis,,,,,,,
no_scaling,-5267.433290,6504.400678,1.00000,0,0,0,0
oxcgrt_floored,-4470.985174,6501.645434,0.78125,0,0,0,0
oxcgrt_independent,-5264.332255,6572.694940,1.00000,0,0,0,0


In [5]:
calib.targets["weekly_deaths"].data.isna().any()

False

In [6]:
calib.targets["weekly_cases"].data.isna()

Date_reported
2022-01-30    False
2022-02-06    False
2022-02-13    False
2022-02-20    False
2022-02-27    False
2022-03-06    False
2022-03-13    False
2022-03-20    False
2022-03-27    False
2022-04-03    False
2022-04-10    False
2022-04-17    False
2022-04-24    False
2022-05-01    False
2022-05-08    False
2022-05-15    False
2022-05-22    False
2022-05-29    False
2022-06-05    False
2022-06-12    False
2022-06-19    False
2022-06-26    False
2022-07-03    False
2022-07-10    False
2022-07-17    False
2022-07-24    False
2022-07-31    False
2022-08-07    False
2022-08-14    False
2022-08-21    False
2022-08-28    False
2022-09-04    False
2022-09-11    False
2022-09-18    False
2022-09-25    False
2022-10-02    False
2022-10-09    False
2022-10-16    False
2022-10-23    False
2022-10-30    False
2022-11-06    False
2022-11-13    False
2022-11-20    False
2022-11-27    False
2022-12-04    False
2022-12-11    False
2022-12-18    False
2022-12-25    False
Name: New_cases, dtype: bo

Finite `log_density` / `nuts_pe` means NUTS can start. NaN or inf in `prop_*` or `n_le0 > 0` on cases/deaths is the usual reason it cannot.
Compare `scale_t0` for floored (should be < 1) against no_scaling and independent (1).